# 06 — E5: format normalization (N1, N2) vs CoT (S5) vs baseline (B0) (W3)

**Default judges (2026-11-14): `gemini-3.8-flash` and `claude-sonnet-4-5`** -- see `config.yaml`.

**Risk flagged by the master plan**: N1 (strip markdown) can destroy structural content (tables, code) on MT-Bench/LLMBar. `strip_markdown()` is a crude regex pass, not a real markdown parser -- the pilot step below prints a few stripped MT-Bench responses so you can eyeball them before running N1 at scale there. It's safe on STYLE (prose-only by construction).

**S5 (CoT) costs more than the others**: max_tokens=300 vs 5 for B0/N1/N2, because it has to write out reasoning before the final verdict line. Budget accordingly (`config.yaml`'s `e5_format_normalization_usd`, default $2).

## Before you run this on Colab
1. Open: `https://colab.research.google.com/github/m00nLeo/judging-the-judges-study/blob/main/notebooks/06_E5_format_normalization.ipynb`
2. Run the setup cell, add `GOOGLE_API_KEY` and `ANTHROPIC_API_KEY` to Colab Secrets.
3. Dry-run -> eyeball N1 stripping on a few MT-Bench pairs -> pilot -> full, one judge + one benchmark at a time.
4. Send back `results/e5_format_normalization/*.jsonl` when done.

In [ ]:
# On Colab, uncomment and run this cell BEFORE the cells below.
# !git clone https://github.com/m00nLeo/judging-the-judges-study.git
# %cd judging-the-judges-study

# !pip install -q anthropic openai google-genai pyyaml
# !bash scripts/fetch_upstream.sh

In [ ]:
import sys
from pathlib import Path

import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "data" / "upstream"
RESULTS_DIR = PROJECT_ROOT / "results" / "e5_format_normalization"

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))
MAX_WORKERS = config["concurrency"]["max_workers"]
E5_CFG = config["sampling"]["e5_format_normalization"]
STRATEGIES = tuple(E5_CFG["strategies"])

from src.experiments.style_bias import load_style_pairs
from src.experiments.position_bias import load_mt_bench_sample_protocol_b
from src.experiments.format_normalization import (
    run_format_normalization_experiment, strip_markdown,
    summarize_style_bias_by_strategy, summarize_agreement_by_strategy,
)
from src.experiments.common import CostModel
from src.judges.clients import build_judge
import json

style_pairs = load_style_pairs(UPSTREAM / "data" / "custom" / "controlled_pairs.jsonl")
mt_bench_pilot = load_mt_bench_sample_protocol_b(
    UPSTREAM / "data" / "raw" / "mt_bench" / "mt_bench.jsonl", n=E5_CFG["mt_bench_pilot_sample"], seed=config["seed"]
)
with open(UPSTREAM / "data" / "raw" / "llmbar" / "llmbar.jsonl", encoding="utf-8") as f:
    llmbar_rows = [json.loads(l) for l in f]
import random
llmbar_pilot = random.Random(config["seed"]).sample(llmbar_rows, min(E5_CFG["llmbar_pilot_sample"], len(llmbar_rows)))
print(f"STYLE: {len(style_pairs)} pairs. MT-Bench pilot: {len(mt_bench_pilot)}. LLMBar pilot: {len(llmbar_pilot)}. Strategies: {STRATEGIES}")

## Eyeball N1 stripping before trusting it on MT-Bench/LLMBar

In [ ]:
for pair in mt_bench_pilot[:2]:
    print("=== ORIGINAL ===\n", pair["response_a"][:500])
    print("\n=== STRIPPED ===\n", strip_markdown(pair["response_a"])[:500])
    print("\n---\n")
print("If stripped output looks mangled (lost tables/code structure), reconsider running N1 on this benchmark's full sample.")

## Step 1 -- dry run (zero cost)

In [ ]:
dry_cache_path = RESULTS_DIR / "_dry_run_cache.jsonl"
dry_cache_path.unlink(missing_ok=True)
dry_judge = build_judge("dry-run", dry_run=True)
cache = run_format_normalization_experiment(style_pairs[:3], dry_judge, judge_name="dry-run", cache_path=dry_cache_path, strategies=STRATEGIES, dry_run=True)
assert len(cache) == 3 * len(STRATEGIES)
print("PASS: pipeline runs end to end with zero API calls.")

## Step 2 -- pilot (REAL calls, small samples, one judge + one benchmark at a time)

In [ ]:
JUDGE_NAME = config["judges"]["e5_format_normalization"][0]  # change index for the other judge
BENCHMARK = "style"  # change to "mt_bench" or "llmbar" to pilot those
PAIRS_BY_BENCHMARK = {"style": style_pairs[:5], "mt_bench": mt_bench_pilot, "llmbar": llmbar_pilot}

price = config["cost_per_1k_tokens"][JUDGE_NAME]
cost_model = CostModel(cost_per_1k_input=price["input"], cost_per_1k_output=price["output"])
budget = config["budget"]["e5_format_normalization_usd"]
judge = build_judge(JUDGE_NAME, dry_run=False)

pilot_cache_path = RESULTS_DIR / f"{JUDGE_NAME}_{BENCHMARK}_pilot.jsonl"
pilot_cache = run_format_normalization_experiment(
    PAIRS_BY_BENCHMARK[BENCHMARK], judge, judge_name=JUDGE_NAME, cache_path=pilot_cache_path,
    strategies=STRATEGIES, cost_model=cost_model, budget_usd=budget, dry_run=False, max_workers=MAX_WORKERS,
)
if BENCHMARK == "style":
    print(summarize_style_bias_by_strategy(PAIRS_BY_BENCHMARK[BENCHMARK], pilot_cache, STRATEGIES))
else:
    print(summarize_agreement_by_strategy(PAIRS_BY_BENCHMARK[BENCHMARK], pilot_cache, STRATEGIES))
print("Review cost above before Step 3 (S5/CoT costs much more than B0/N1/N2).")

## Step 3 -- full (only after reviewing pilot cost)

In [ ]:
if BENCHMARK == "style":
    full_pairs = style_pairs
elif BENCHMARK == "mt_bench":
    full_pairs = load_mt_bench_sample_protocol_b(
        UPSTREAM / "data" / "raw" / "mt_bench" / "mt_bench.jsonl", n=E5_CFG["mt_bench_full_sample"], seed=config["seed"]
    )
else:
    full_pairs = random.Random(config["seed"]).sample(llmbar_rows, min(E5_CFG["llmbar_full_sample"], len(llmbar_rows)))

full_cache_path = RESULTS_DIR / f"{JUDGE_NAME}_{BENCHMARK}_full.jsonl"
full_cache = run_format_normalization_experiment(
    full_pairs, judge, judge_name=JUDGE_NAME, cache_path=full_cache_path,
    strategies=STRATEGIES, cost_model=cost_model, budget_usd=budget, dry_run=False, max_workers=MAX_WORKERS,
)
if BENCHMARK == "style":
    result = summarize_style_bias_by_strategy(full_pairs, full_cache, STRATEGIES)
else:
    result = summarize_agreement_by_strategy(full_pairs, full_cache, STRATEGIES)
print(result)
print(f"\nRepeat Steps 2-3 for the other judge and the other benchmarks (BENCHMARK = 'mt_bench', 'llmbar').")
print("When all combinations are done, send back results/e5_format_normalization/*_full.jsonl.")